In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.14 Quasiparticles, Spectral Functions, and GW

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.14",
    title="Quasiparticles, Spectral Functions, and GW",
    blurb="What survives of 'an electron's energy' when electrons interact? "
    "The Lehmann spectral function — built exactly from the sectors of §8.13, "
    "sum rules landing at 1.0000000000 and the first moment at ε(k) + U/2 "
    "to ten digits — answers: a quasiparticle peak with weight Z < 1, "
    "plus satellites carrying the rest. The dimer's self-energy is "
    "extracted from Dyson's equation, Z computed two independent ways, "
    "and then the workhorse of modern band theory is built from scratch: "
    "G₀W₀, screened interaction and all, whose quasiparticle gap rescues "
    "Hartree–Fock's at weak coupling (−19% → −2.6% at U = 2t) and "
    "honestly collapses in the Mott regime. The many-body deficit of "
    "the §8.11 silicon gap, explained by construction.",
    difficulty="advanced",
    estimate="150–180 min",
)

## Notebook overview

Every band structure this volume has drawn —
[§8.9](tight-binding.ipynb)'s cosines, [§8.11](epm-band-structures.ipynb)'s
silicon — assigned each electron an energy $\varepsilon(k)$ as if the
others were scenery. [§8.13](hubbard-model.ipynb) showed what that misses.
But photoemission experiments *do* measure sharp band-like dispersions in
real interacting materials, and [§8.8](exact-conditions-band-gap.ipynb)
showed the fundamental gap is a difference of $N\pm1$ ground-state
energies. This notebook builds the object that reconciles all of it: the
one-particle **spectral function** $A(k, \omega)$ — the probability
distribution of energies for adding or removing one electron — which is
what [§7.24](../07-quantum-statistical-mechanics/greens-functions.ipynb)'s
Green function becomes when the many-body machinery of
[§8.13](hubbard-model.ipynb) evaluates it exactly.

The build: the Lehmann representation computed literally — every $N\pm1$
eigenstate of the $L = 6$ Hubbard ring, every matrix element of
$\hat c^\dagger_k$ — with its two exact sum rules landing not
approximately but to ten digits (total weight $1$; first moment
$\varepsilon(k) + U/2$, interactions shifting the band's center of
gravity by a constant and nothing else). The Hubbard dimer then becomes
the Rosetta stone: its Green function has *two* poles per orbital where
the band picture promises one — a quasiparticle carrying weight
$Z = \tfrac12 + 2t/\sqrt{U^2 + 16t^2}$ (a closed form our numerics must
and does hit) and a satellite carrying $1 - Z$. Dyson's equation is run
*backwards* to extract the self-energy $\Sigma(\omega)$, whose single
pole is where all the correlation hides, and $Z$ is recomputed from
$\partial\Sigma/\partial\omega$ — two unrelated routes, one number. The
chain's $A(k, \omega)$ then shows the full drama: a sharp band at
$U = 0$ fractioning into upper and lower Hubbard bands, with the
spectral gap *equal* to [§8.13](hubbard-model.ipynb)'s charge gap by
construction. The finale builds Hedin's $G_0W_0$ {cite}`hedin1965` on
the dimer — polarizability, plasmon-pole screened interaction
$W(\omega)$, correlation self-energy, quasiparticle equation — and holds
it against the exact solution in the spirit of Romaniello's dimer
autopsy {cite}`romaniello2009`: the gap error shrinks from
Hartree–Fock's $-19\%$ to $-2.6\%$ at $U = 2t$, and grows to $-38\%$ by
$U = 8t$ — GW is the world's production band-gap machine *and* a
weak-coupling theory, both facts measured here. This is the machinery
behind every "$GW$ band structure" in the literature — including the
photocathode calculations of the author's thesis {cite}`amador2019` —
and it is why [§8.11](epm-band-structures.ipynb)'s silicon gap came out
$0.35$ eV short.

> **Conventions (this notebook).** Hopping $t = 1$ sets the energy unit.
> Chains are periodic ($L = 6$, half filled); the dimer is open with one
> electron per spin. Energies are measured from the $N$-electron ground
> state (no chemical-potential shift: addition poles at
> $E_n^{N+1} - E_0^N$, removal poles at $E_0^N - E_n^{N-1}$), so
> particle–hole symmetry puts the gap's center at $U/2$. All spectra are
> kept as exact pole lists (energies, weights) from dense
> `numpy.linalg.eigh` of the [§8.13](hubbard-model.ipynb) sector
> machinery; Lorentzian broadening $\eta$ is applied only for plotting.
> Momentum operators are $\hat c^\dagger_{k\sigma} = L^{-1/2}\sum_j
> e^{ikj}\hat c^\dagger_{j\sigma}$ on the ring, $k = 2\pi m/L$.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: an exact sum rule, a closed-form pole
> weight, an identity between two routes to the same number. A ✓ is
> strong evidence; a ✗ is a prompt to locate the discrepancy, not an
> automatic verdict.
>
> **Scope.** Zero temperature, one dimension, exact diagonalization, and
> the $G_0W_0$ approximation on a two-site model. Production GW — plane
> waves, frequency integration, self-consistency flavors — is reviewed
> in Onida, Reining & Rubio {cite}`onida2002` and Martin
> {cite}`martin2004`; what light does with these quasiparticles (and the
> electron–hole attraction GW leaves out) is
> [§8.15](optics-excitons.ipynb)'s story.

## Theory in brief

### The Lehmann representation: dynamics from eigenstates

The retarded Green function of
[§7.24](../07-quantum-statistical-mechanics/greens-functions.ipynb),
$G_k(\omega)$, asks: inject an electron with momentum $k$ into the
interacting ground state — with what amplitude does the system carry it
at frequency $\omega$? Inserting a complete set of $N\pm1$ eigenstates
turns the question into spectroscopy:

```{math}
:label: eq-qp-lehmann
A_k(\omega) \;=\; \sum_n \big|\langle n^{N+1}|\,\hat c^\dagger_k\,
|0^N\rangle\big|^2\,\delta\!\big(\omega - (E_n^{N+1} - E_0^N)\big)
\;+\; \sum_m \big|\langle m^{N-1}|\,\hat c_k\,|0^N\rangle\big|^2\,
\delta\!\big(\omega - (E_0^N - E_m^{N-1})\big) .
```

Photoemission measures the second sum; inverse photoemission the first.
Completeness of the $N\pm1$ eigenbases forces the **zeroth-moment sum
rule** $\int A_k\,d\omega = \{\hat c_k, \hat c^\dagger_k\} = 1$, and the
equation-of-motion anticommutator
$\{[\hat c_k, \hat H], \hat c^\dagger_k\}$ gives the **first moment**
$\int \omega A_k\,d\omega = \varepsilon(k) + U\langle n_{-\sigma}\rangle$:
whatever interactions do to the *shape*, the band's center of gravity
only shifts by the Hartree constant. Both are exact at any $U$, and both
are gates below.

### Quasiparticles, Z, and the self-energy

For free electrons $A_k(\omega) = \delta(\omega - \varepsilon_k)$: one
pole, weight $1$. Interactions split the delta into a **quasiparticle**
peak — the surviving band-like excitation, carrying weight $Z_k < 1$ —
and incoherent **satellites** carrying $1 - Z_k$ (the injected electron
shakes up the others, and part of its identity goes into the
shake-up). The bookkeeping device is the self-energy, defined by Dyson's
equation

```{math}
:label: eq-qp-dyson
G_k(\omega) \;=\; \frac{1}{\omega - \varepsilon_k^{\mathrm{HF}}
- \Sigma_k(\omega)},
\qquad
Z \;=\; \Big(1 - \frac{\partial\,\mathrm{Re}\,\Sigma}{\partial\omega}
\Big)^{-1}_{\omega = E_{\mathrm{qp}}} ,
```

which this notebook runs *backwards*: with $G$ exact from Lehmann,
$\Sigma = \omega - \varepsilon^{\mathrm{HF}} - G^{-1}$ is measured, not
approximated. For the half-filled dimer everything is closed-form: the
bonding orbital's two poles sit at
$E_0 + t$ and $U + t - E_0$ (with $E_0$ from Eq. {eq}`eq-hub-dimer` of
[§8.13](hubbard-model.ipynb)), with quasiparticle weight

```{math}
:label: eq-qp-z
Z \;=\; \frac12 + \frac{2t}{\sqrt{U^2 + 16t^2}} ,
```

and $\Sigma(\omega)$ has exactly one pole — correlation compressed into
one resonance.

### GW: screening the exchange

Hartree–Fock fails for gaps because its exchange is *bare*: in a solid,
the other electrons rearrange to screen any added charge. Hedin's
expansion {cite}`hedin1965` replaces the bare interaction $v$ in the
exchange diagram by the **screened** one,

```{math}
:label: eq-qp-gw
W(\omega) \;=\; \frac{v}{1 - v\,\chi_0(\omega)},
\qquad
\Sigma^{GW} \;=\; \mathrm{i}\,G\,W ,
```

with $\chi_0$ the independent-particle polarizability. On the dimer,
$\chi_0$ has a single resonance at the HF gap $\Delta = 2t$, so $W$ has
a single "plasmon" pole at $\omega_p = \sqrt{\Delta^2 + 2\Delta U}$ —
screening *stiffens* the resonance — and the correlation self-energy is
a one-pole function whose quasiparticle equation
$\omega = \varepsilon^{\mathrm{HF}} + \Sigma_c(\omega)$ closes with
`scipy.optimize.brentq`. Everything in Eq. {eq}`eq-qp-gw` is built
explicitly in Exercise 5, and judged against the exact dimer in the
spirit of {cite}`romaniello2009`.

## Setup

In [ ]:
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

T_HOP = 1.0  # the hopping integral: the energy unit throughout


def jw_sign(mask, i):
    """Jordan-Wigner sign (-1)^(occupied modes below bit i); see section 8.13.

    Parameters
    ----------
    mask : int
        Occupation bit mask.
    i : int
        Mode index acted on.

    Returns
    -------
    float
        +1.0 or -1.0.
    """
    return -1.0 if (mask & ((1 << i) - 1)).bit_count() & 1 else 1.0


def basis_masks(n_modes, n_occ):
    """All n_modes-bit masks with n_occ bits set (the section 8.13 sector basis).

    Parameters
    ----------
    n_modes, n_occ : int
        Orbital count and occupation.

    Returns
    -------
    list of int
        The masks in canonical order.
    """
    return [sum(1 << i for i in combo) for combo in combinations(range(n_modes), n_occ)]


def hubbard_dense(n_sites, n_up, n_dn, u_int, pbc=True):
    """Dense Hubbard sector Hamiltonian (small chains; the section 8.13 build).

    Parameters
    ----------
    n_sites, n_up, n_dn : int
        Chain length and electron numbers.
    u_int : float
        On-site repulsion.
    pbc : bool, optional
        Periodic if True.

    Returns
    -------
    tuple
        (H dense array, up masks, down masks).
    """
    ups = basis_masks(n_sites, n_up)
    dns = basis_masks(n_sites, n_dn)
    up_index = {m: a for a, m in enumerate(ups)}
    dn_index = {m: b for b, m in enumerate(dns)}
    dim_dn = len(dns)
    ham = np.zeros((len(ups) * dim_dn, len(ups) * dim_dn))
    for a, mask_up in enumerate(ups):
        for b, mask_dn in enumerate(dns):
            ham[a * dim_dn + b, a * dim_dn + b] = (
                u_int * (mask_up & mask_dn).bit_count()
            )
    bonds = [(i, (i + 1) % n_sites) for i in range(n_sites if pbc else n_sites - 1)]

    def spin_hops(masks, index):
        """(from, to, sign) hopping triples within one spin species."""
        moves = []
        for a, mask in enumerate(masks):
            for i, j in bonds:
                for src, dst in ((i, j), (j, i)):
                    if (mask >> src) & 1 and not (mask >> dst) & 1:
                        lo, hi = min(src, dst), max(src, dst)
                        between = (
                            mask & ((1 << hi) - 1) & ~((1 << (lo + 1)) - 1)
                        ).bit_count()
                        moves.append(
                            (
                                a,
                                index[mask ^ (1 << src) ^ (1 << dst)],
                                (-1.0) ** between,
                            )
                        )
        return moves

    for a, a2, sign in spin_hops(ups, up_index):
        for b in range(dim_dn):
            ham[a2 * dim_dn + b, a * dim_dn + b] += -T_HOP * sign
    for b, b2, sign in spin_hops(dns, dn_index):
        for a in range(len(ups)):
            ham[a * dim_dn + b2, a * dim_dn + b] += -T_HOP * sign
    return ham, ups, dns


def cdag_up_matrix(i, ups_from, dns, ups_to):
    """Matrix of cdag_{i,up} between fixed-(N_up, N_dn) sectors.

    Parameters
    ----------
    i : int
        Site index.
    ups_from, ups_to : list of int
        Up-spin masks of the source and target sectors.
    dns : list of int
        Down-spin masks (unchanged spectator).

    Returns
    -------
    numpy.ndarray
        The rectangular operator matrix.
    """
    to_index = {m: a for a, m in enumerate(ups_to)}
    dim_dn = len(dns)
    mat = np.zeros((len(ups_to) * dim_dn, len(ups_from) * dim_dn))
    for a, mask in enumerate(ups_from):
        if not (mask >> i) & 1:
            a2 = to_index[mask | (1 << i)]
            sign = jw_sign(mask, i)
            for b in range(dim_dn):
                mat[a2 * dim_dn + b, a * dim_dn + b] = sign
    return mat


def c_up_matrix(i, ups_from, dns, ups_to):
    """Matrix of c_{i,up} between fixed-(N_up, N_dn) sectors.

    Parameters
    ----------
    i : int
        Site index.
    ups_from, ups_to : list of int
        Up-spin masks of source and target sectors.
    dns : list of int
        Down-spin masks.

    Returns
    -------
    numpy.ndarray
        The rectangular operator matrix.
    """
    to_index = {m: a for a, m in enumerate(ups_to)}
    dim_dn = len(dns)
    mat = np.zeros((len(ups_to) * dim_dn, len(ups_from) * dim_dn))
    for a, mask in enumerate(ups_from):
        if (mask >> i) & 1:
            a2 = to_index[mask ^ (1 << i)]
            sign = jw_sign(mask, i)
            for b in range(dim_dn):
                mat[a2 * dim_dn + b, a * dim_dn + b] = sign
    return mat


def lehmann_spectra(n_sites, u_int, pbc=True):
    """Exact Lehmann pole data for every momentum of the half-filled chain.

    Diagonalizes the N, N+1, and N-1 sectors densely, forms
    cdag_k = L^{-1/2} sum_j e^{ikj} cdag_j, and collects Eq.
    (eq-qp-lehmann)'s poles and weights exactly (no broadening).

    Parameters
    ----------
    n_sites : int
        Chain length L (even).
    u_int : float
        On-site repulsion.
    pbc : bool, optional
        Periodic if True.

    Returns
    -------
    dict
        m -> (addition energies, addition weights, removal energies,
        removal weights) for k = 2 pi m / L.
    """
    n_half = n_sites // 2
    ham_n, ups_n, dns_n = hubbard_dense(n_sites, n_half, n_half, u_int, pbc)
    vals_n, vecs_n = np.linalg.eigh(ham_n)
    e_ground, ground = vals_n[0], vecs_n[:, 0]
    ham_p, ups_p, _ = hubbard_dense(n_sites, n_half + 1, n_half, u_int, pbc)
    vals_p, vecs_p = np.linalg.eigh(ham_p)
    ham_m, ups_m, _ = hubbard_dense(n_sites, n_half - 1, n_half, u_int, pbc)
    vals_m, vecs_m = np.linalg.eigh(ham_m)
    cdag_site = [cdag_up_matrix(i, ups_n, dns_n, ups_p) for i in range(n_sites)]
    c_site = [c_up_matrix(i, ups_n, dns_n, ups_m) for i in range(n_sites)]
    spectra = {}
    for m_k in range(n_sites):
        k = 2.0 * np.pi * m_k / n_sites
        phases = np.exp(1j * k * np.arange(n_sites)) / np.sqrt(n_sites)
        cdag_k = sum(p * mat for p, mat in zip(phases, cdag_site))
        c_k = sum(np.conj(p) * mat for p, mat in zip(phases, c_site))
        add_amp = vecs_p.T @ (cdag_k @ ground)
        rem_amp = vecs_m.T @ (c_k @ ground)
        spectra[m_k] = (
            vals_p - e_ground,
            np.abs(add_amp) ** 2,
            e_ground - vals_m,
            np.abs(rem_amp) ** 2,
        )
    return spectra


def broaden(energies, weights, w_grid, eta):
    """Lorentzian-broadened spectrum from exact poles (plotting only).

    Parameters
    ----------
    energies, weights : numpy.ndarray
        Pole positions and weights.
    w_grid : numpy.ndarray
        Frequency grid.
    eta : float
        Lorentzian half-width.

    Returns
    -------
    numpy.ndarray
        The broadened A(w) on the grid.
    """
    return (
        weights[:, None]
        * eta
        / np.pi
        / ((w_grid[None, :] - energies[:, None]) ** 2 + eta**2)
    ).sum(axis=0)

## Exercise 1 — The Lehmann machine, certified where the answer is known

Before trusting Eq. {eq}`eq-qp-lehmann` on an interacting system, it
must reproduce the one case where "the energy of an electron" is
unambiguous.

**Part a)** Build the full Lehmann data for the $L = 6$ ring at $U = 0$
with `lehmann_spectra`. For each $k$, the spectrum must be a *single*
pole — one distinct energy carrying the full unit weight — at exactly
the tight-binding energy $\varepsilon(k) = -2t\cos k$ of
[§8.9](tight-binding.ipynb). (On the doubly degenerate levels
$\varepsilon = \pm t$ the diagonalizer may split that weight across two
degenerate $N\pm1$ eigenstates, so count *distinct* energies.) Sector
eigenstates, operator matrices, Jordan–Wigner signs, and momentum
transform all conspire to reproduce six numbers known in advance.

**Part b)** Check the zeroth-moment sum rule at $U = 0$ *and* $U = 4$:
the weights at every $k$ must total exactly $1$ — not because the
physics is simple (at $U = 4$ each $k$ has dozens of poles) but because
the $N\pm1$ eigenbases are complete. A sum rule that holds at
$10^{-10}$ under strong interaction is the machine's certificate.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the machine's certificate

Free poles on the tight-binding band at $10^{-10}$; total weight $1$ at
every momentum, interacting or not.

In [ ]:
validate.check(
    bool(pole_dev < 1e-10),
    "U = 0 spectra collapse onto eps(k) = -2t cos k",
    f"max deviation {pole_dev:.1e}",
)
validate.check(
    bool(weight_dev_u0 < 1e-10 and weight_dev_u4 < 1e-10),
    "zeroth-moment sum rule = 1 at U = 0 and U = 4, all k",
    f"max deviations {weight_dev_u0:.1e}, {weight_dev_u4:.1e}",
)

## Exercise 2 — The dimer's Green function: one band state, two poles

The smallest system where "the electron's energy" stops being one
number.

**Part a)** Compute the exact Lehmann spectrum of the half-filled dimer
(open, $L = 2$; the $k = 0$ combination is the bonding orbital, $k =
\pi$ antibonding) at $U = 4$. The bonding orbital must show exactly two
poles: the quasiparticle (electron *removal* at $E_0 + t = 0.1716$,
weight $0.8536$) and a satellite (electron *addition* at $U + t - E_0 =
5.8284$, weight $0.1464$) — the removed electron sometimes leaves its
partner shaken into the antibonding level.

**Part b)** Gate the weights against the closed form, Eq.
{eq}`eq-qp-z`: $Z = \tfrac12 + 2t/\sqrt{U^2 + 16t^2}$ at $U = 2, 4, 8$
(three closed forms, $10^{-12}$), satellite weight $1 - Z$ by the sum
rule.

**Part c)** Plot $A_{\mathrm{bonding}}(\omega)$ at $U = 1, 4, 8$: watch
the quasiparticle surrender weight to the satellite as correlation
grows — the single free pole ($Z = 1$) dying by degrees.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — the closed forms

Both poles on their closed-form positions, and $Z$ on Eq. {eq}`eq-qp-z`
at three couplings.

In [ ]:
validate.close(
    float(rem_e[0]), e_ground + 1.0, "quasiparticle pole at E0 + t", rtol=1e-10
)
validate.close(
    float(add_e[0]), 5.0 - e_ground, "satellite pole at U + t - E0", rtol=1e-10
)
validate.check(
    bool(z_dev < 1e-12),
    "Z = 1/2 + 2t/sqrt(U^2 + 16t^2) at U = 2, 4, 8",
    f"worst deviation {z_dev:.1e}",
)
validate.close(
    float(add_w[0] + rem_w[0]),
    1.0,
    "quasiparticle + satellite = one electron",
    rtol=1e-10,
)

## Exercise 3 — Dyson run backwards: measuring the self-energy

The self-energy is usually something one *approximates*; on the dimer
it can be *measured*.

**Part a)** From the exact two-pole $G_{\mathrm{bonding}}(\omega)$ at
$U = 4$ (a real rational function away from its poles), extract
$\Sigma(\omega) = \omega - \varepsilon^{\mathrm{HF}} - 1/G(\omega)$ on a
real grid, with $\varepsilon^{\mathrm{HF}} = -t + U/2$ the Hartree–Fock
level. Plot $G^{-1}$'s ingredients and $\Sigma$: a two-pole $G$ forces a
*one-pole* $\Sigma$ — all correlation compressed into a single
resonance between the quasiparticle and the satellite.

**Part b)** Recompute the quasiparticle weight from the *derivative*
route of Eq. {eq}`eq-qp-dyson`, $Z = (1 - \partial\Sigma/\partial
\omega)^{-1}$ at $\omega = E_{\mathrm{qp}}$ (central differences on the
measured $\Sigma$), and check it against the *pole-weight* route of
Exercise 2 at $U = 2, 4, 8$: two unrelated definitions, one number, at
$10^{-7}$.

**Part c)** Confirm $\max_\omega|\Sigma|$ on a gap-centered window dies
as $U \to 0$ (measure at $U = 0.5, 0.25$: the ratio must be $\approx 4$,
since $\Sigma \propto U^2$ — correlation is second order, as
[§8.13](hubbard-model.ipynb)'s superexchange already knew).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3 — one number, two routes, second order

The derivative-route $Z$ against the pole-weight $Z$ at three
couplings, and the $U^2$ scaling of the measured $\Sigma$.

In [ ]:
validate.check(
    bool(z_route_dev < 1e-7),
    "Z from dSigma/dw = Z from the pole weight (U = 2, 4, 8)",
    f"worst disagreement {z_route_dev:.1e}",
)
validate.close(
    scale_ratio, 4.0, "Sigma scales as U^2 (correlation is second order)", rtol=5e-2
)

## Exercise 4 — $A(k, \omega)$: watching a band die into Hubbard bands

The chain's full momentum-resolved spectral function — what an ARPES
beamline would see, computed exactly.

**Part a)** Assemble $A(k, \omega)$ heatmaps for the $L = 6$ ring at
$U = 0, 2, 4, 8$ from the exact pole data (Lorentzian broadening for
display only). At $U = 0$: the tight-binding cosine. As $U$ grows: the
band splits into *lower and upper Hubbard bands* separated by the
correlation gap, each carrying part of every $k$'s weight.

**Part b)** Check the first-moment sum rule at $U = 4$ and $8$: for
every $k$, $\int\omega A_k\,d\omega = \varepsilon(k) + U/2$ *to ten
digits* — the interactions rearrange the weight but cannot move its
center of gravity (an operator identity, and a merciless test of every
matrix element).

**Part c)** Measure the spectral gap (lowest addition pole minus
highest removal pole with nonzero weight, over all $k$) at $U = 4$ and
confirm it *equals* the charge gap $E_0^{N+1} + E_0^{N-1} - 2E_0^N$ of
[§8.13](hubbard-model.ipynb) at $10^{-10}$: photoemission and
thermodynamics agree by construction, because they are the same
eigenvalues.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — sum rules with no mercy

First moment at $\varepsilon(k) + U/2$ over twelve $(U, k)$ pairs;
spectral gap $=$ charge gap.

In [ ]:
validate.check(
    bool(moment_dev < 1e-9),
    "first moment = eps(k) + U/2 at U = 4, 8, all k",
    f"worst deviation {moment_dev:.1e}",
)
validate.close(
    spec_gap,
    float(charge_gap),
    "spectral gap = charge gap (same eigenvalues)",
    rtol=1e-10,
)

## Exercise 5 — Verdict: $G_0W_0$ built from scratch, judged exactly

The finale assembles the machinery behind essentially every published
*ab initio* quasiparticle band structure — on the one system where the
exact answer sits alongside it.

**Part a)** Build the ingredients of Eq. {eq}`eq-qp-gw` for the dimer.
HF: $\varepsilon_{b,a} = \mp t + U/2$, gap $\Delta = 2t$ at *every* $U$
(bare exchange cannot see on-site correlation; on-site $U$ has no
opposite-spin exchange). Polarizability: one resonance,
$\chi_0(\omega) = 2\Delta c_0/(\omega^2 - \Delta^2)$ with $c_0 = 1$ from
the bonding→antibonding transition densities $(\pm\tfrac12)$. Screening:
the RPA denominator $1 - U\chi_0$ shifts the pole to $\omega_p =
\sqrt{\Delta^2 + 2\Delta U}$, and $W$ acquires strength $\lambda =
U^2\Delta/\omega_p$. Correlation self-energy: the plasmon-pole formula
gives one pole per orbital, $\Sigma_c^{b}(\omega) = \lambda/2 \cdot
(\omega - \varepsilon_a - \omega_p)^{-1}$ (and mirrored for $a$).

**Part b)** Solve the quasiparticle equation $\omega =
\varepsilon^{\mathrm{HF}} + \Sigma_c(\omega)$ with
`scipy.optimize.brentq` for both orbitals at $U/t = 0.5, 1, 2, 4, 8$,
and print the verdict table: exact gap (from Exercise 2's Lehmann
poles), HF gap, $G_0W_0$ gap, and both errors. The pattern to gate: GW
*always* beats HF; at $U \leq 2t$ it is a few-percent theory
($+0.7\%$, $+1.0\%$, $-2.6\%$); by $U = 8t$ it is off by $-38\%$ —
screening builds a better weak-coupling theory, not a Mott theory
{cite}`romaniello2009`. The GW quasiparticle weight $Z_b$ falls
$0.995 \to 0.887$: correlation eating the quasiparticle, now predicted
rather than measured.

**Part c)** Close the volume-wide loop: this is why
[§8.11](epm-band-structures.ipynb)'s converged EPM silicon still sat
$0.35$ eV below the measured gap, and why production codes run
$G_0W_0$ *on top of* the DFT bands of [§8.8](exact-conditions-band-gap.ipynb)
— as in the alkali-antimonide photocathode calculations of the author's
thesis {cite}`amador2019`, where exactly this machinery (plane-wave
basis instead of two sites) turns Kohn–Sham bands into measurable
quasiparticle energies.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — a weak-coupling theory, measured as such

GW strictly better than HF at every coupling; a few-percent theory
through $U = 2t$; honestly wrong by tens of percent at $U = 8t$; $Z$
and $\omega_p$ behaving as screening demands.

In [ ]:
validate.check(
    bool(all(abs(r["err_gw"]) < abs(r["err_hf"]) for r in rows)),
    "G0W0 beats Hartree-Fock at every U",
    "all five couplings",
)
small_u_errors = ", ".join(f"{100 * row['err_gw']:+.1f}%" for row in rows[:3])
validate.check(
    bool(all(abs(r["err_gw"]) < 0.03 for r in rows if r["u"] <= 2.0)),
    "G0W0 gap within 3% for U <= 2t",
    f"errors {small_u_errors}",
)
validate.close(
    float(rows[-1]["err_gw"]), -0.38, "the Mott failure, quantified (U = 8t)", atol=0.03
)
validate.check(
    bool(
        all(r1["z_b"] > r2["z_b"] for r1, r2 in zip(rows, rows[1:]))
        and rows[-1]["z_b"] > 0.85
    ),
    "Z_b falls monotonically (0.995 -> 0.887)",
    f"{rows[0]['z_b']:.3f} -> {rows[-1]['z_b']:.3f}",
)
validate.close(
    dimer_g0w0(4.0)["omega_p"],
    float(np.sqrt(4.0 + 16.0)),
    "plasmon pole at sqrt(Delta^2 + 2 Delta U)",
    rtol=1e-12,
)

:::{admonition} With your assistant
:class: tip
The satellites of Exercise 4 hide a clean strong-coupling law: at large
$U$ the lower and upper Hubbard bands are separated by $\approx U$
(each still roughly $4t$ wide). Have your assistant compute the
weight-averaged center of the addition spectrum and of the removal
spectrum (over all $k$) for the $L = 6$ ring at $U = 12$ and $16$. Then
run the check that is yours alone: the center-to-center separation must
track $U$ within one hopping unit at both couplings
(`numpy.isclose`, `atol=1.0`) — and, as a bonus, the first-moment sum
rule forces the *midpoint* of the two centers to sit at $U/2$ exactly.
The check is yours.
:::

## Notebook summary

The volume's language shifted from energies to spectra, with every step
certified. The Lehmann machinery — [§8.13](hubbard-model.ipynb)'s
sectors plus momentum-resolved $\hat c^\dagger_k$ matrices — reproduced
the free chain's band to $10^{-10}$ and obeyed its two exact sum rules
under full interaction: total weight $1.0000000000$ at every momentum,
first moment $\varepsilon(k) + U/2$ to ten digits at $U = 4$ and $8$.
The dimer showed what interactions do to "the electron's energy": two
poles per orbital, a quasiparticle with closed-form weight
$Z = \tfrac12 + 2t/\sqrt{U^2 + 16t^2}$ (hit at $10^{-13}$ three times)
and a satellite with the rest. Dyson's equation, run backwards, measured
the self-energy — one pole at $U/2 + 3t$, weight scaling as $U^2$
(ratio $3.95$ measured against the prediction of $4$) — and the derivative route to $Z$ agreed with
the pole-weight route at $10^{-8}$. The chain's $A(k, \omega)$ showed
the cosine band dying into Hubbard bands with the spectral gap equal to
the charge gap at $10^{-10}$. And $G_0W_0$, assembled from
polarizability to plasmon pole ($\omega_p = \sqrt{\Delta^2 + 2\Delta U}$)
to quasiparticle equation, was judged against the exact dimer: better
than Hartree–Fock at every coupling, a few-percent theory through
$U = 2t$, and $-38\%$ wrong at $U = 8t$ — the production band-gap
machine of modern electronic structure, certified as the weak-coupling
theory it is. [§8.11](epm-band-structures.ipynb)'s missing $0.35$ eV
now has a name, a diagram, and a price tag.

## Outlook

- $A(k, \omega)$ is what photoemission sees; what *optics* sees is a
  two-particle story — the excited electron and the hole it left behind
  attract, and the resulting excitons live *inside* the quasiparticle
  gap. [§8.15](optics-excitons.ipynb) builds the dielectric function on
  [§8.11](epm-band-structures.ipynb)'s EPM machinery and pulls the
  exciton out of a model Bethe–Salpeter equation.
- The plasmon-pole $G_0W_0$ here becomes production GW by swapping two
  sites for plane waves and one resonance for a frequency integration —
  Onida, Reining & Rubio {cite}`onida2002` is the standard tour;
  self-consistency variants (ev$GW$, QS$GW$) start from the same
  $\Sigma = \mathrm{i}GW$.
- Where GW died ($U \gg t$), dynamical mean-field theory takes over —
  built exactly on this notebook's objects ($A(\omega)$, $\Sigma(\omega)$)
  with the Hubbard dimer's big sibling, an impurity model, at its core;
  Martin, Reining & Ceperley {cite}`martinreining2016` Ch. 21 connects the threads.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()